# Part 26b: sklearn Pipeline -- Hyperparameter Search and Evaluation

**Dataset:** Hotel Bookings (same as Part 26a)

**Covers sections 6-9:** hyperparameter search strategies, Bayesian optimisation with Optuna, feature importance, final evaluation and model card.

Run the collapsed setup section below before executing any other cell.


## 0. Setup

Re-runs imports, data loading, and pipeline construction from Part 26a so this notebook is self-contained.

In [ ]:
from __future__ import annotations

# stdlib
from pathlib import Path
from typing import Any

# data
import numpy as np

# hyperparameter search
import optuna
import pandas as pd
from scipy.stats import loguniform, uniform

# sklearn -- transformers and composition
from sklearn.compose import ColumnTransformer

# sklearn -- models
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# sklearn -- metrics
from sklearn.metrics import (
    classification_report,
    f1_score,
)

# sklearn -- model selection
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

# persistence
import shutil

# tables
from great_tables import GT, md as gt_md
import joblib

# visualisation
from lets_plot import (
    LetsPlot,
    aes,
    as_discrete,
    geom_bar,
    geom_line,
    geom_point,
    ggplot,
    labs,
    scale_color_manual,
    scale_fill_manual,
)

# logging
from loguru import logger

# project branding
from ark.plot.gt_style import metrics_report, themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import AI_ACCENT, BRAND_PALETTE, DANGER, INFO, PRIMARY, SUCCESS, WARNING

LetsPlot.setup_html()

In [ ]:
DATA_URL: str = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
DATA_PATH: Path = Path("data/hotel_bookings.csv")
MODEL_DIR: Path = Path("models")
RANDOM_STATE: int = 42

In [ ]:
NUMERICAL_FEATURES: list[str] = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "previous_cancellations",
    "booking_changes",
    "total_of_special_requests",
    "days_in_waiting_list",
]
CATEGORICAL_FEATURES: list[str] = [
    "hotel",
    "meal",
    "market_segment",
    "distribution_channel",
    "customer_type",
    "deposit_type",
]
ALL_FEATURES: list[str] = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
TARGET_CLF: str = "is_canceled"

In [ ]:
def load_hotel_data(
    path: Path = DATA_PATH,
    url: str = DATA_URL,
) -> pd.DataFrame:
    """Load hotel bookings, downloading from url on first call."""
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        raw = pd.read_csv(url)
        raw.to_csv(path, index=False)
        logger.success(f"Downloaded {len(raw):,} rows to {path}")
    else:
        raw = pd.read_csv(path)
        logger.info(f"Loaded {len(raw):,} rows from cache: {path}")
    df = raw[(raw["adr"] > 0) & (raw["adr"] <= 1000)].reset_index(drop=True)
    logger.info(f"After cleaning: {len(df):,} rows")
    return df

In [ ]:
df: pd.DataFrame = load_hotel_data()
logger.info(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
X: pd.DataFrame = df[ALL_FEATURES].copy()
y: np.ndarray = df[TARGET_CLF].to_numpy(dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)
logger.info(f"Train: {X_train.shape}  Test: {X_test.shape}")
logger.info(f"Train cancel rate: {y_train.mean():.2%}  Test cancel rate: {y_test.mean():.2%}")

In [ ]:
numeric_transformer: Pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

In [ ]:
categorical_transformer: Pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)

In [ ]:
preprocessor: ColumnTransformer = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERICAL_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

In [ ]:
pipe_full: Pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

In [ ]:
pipe_full.fit(X_train, y_train)
print("Pipeline refitted for Part 26b.")

In [ ]:
skf: StratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores: np.ndarray = cross_val_score(
    pipe_full,
    X_train,
    y_train,
    cv=skf,
    scoring="f1",
    n_jobs=-1,
)
logger.info(f"CV F1 per fold: {cv_scores.round(3)}")
logger.info(f"Mean +/- Std: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

## 6. Hyperparameter Search

**From Part 24 (ML Workflow, Section 7):** hyperparameters are not learned from data -- they control learning. Finding good values requires a systematic search.

We compare three strategies:

<table style="width:100%;border-collapse:collapse;font-family:system-ui,sans-serif;font-size:0.9rem;margin:16px 0">
  <thead>
    <tr style="background:#0369A1;color:#fff">
      <th style="padding:10px 14px;text-align:left;font-weight:600">Strategy</th>
      <th style="padding:10px 14px;text-align:left;font-weight:600">Coverage</th>
      <th style="padding:10px 14px;text-align:left;font-weight:600">Cost</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background:#F8FAFC">
      <td style="padding:9px 14px;border-bottom:1px solid #E2E8F0"><code>GridSearchCV</code></td>
      <td style="padding:9px 14px;border-bottom:1px solid #E2E8F0">Exhaustive grid</td>
      <td style="padding:9px 14px;border-bottom:1px solid #E2E8F0">Expensive for many params</td>
    </tr>
    <tr style="background:#fff">
      <td style="padding:9px 14px;border-bottom:1px solid #E2E8F0"><code>RandomizedSearchCV</code></td>
      <td style="padding:9px 14px;border-bottom:1px solid #E2E8F0">Sample from distributions</td>
      <td style="padding:9px 14px;border-bottom:1px solid #E2E8F0">Fixed budget, broader space</td>
    </tr>
    <tr style="background:#F8FAFC">
      <td style="padding:9px 14px"><code>Optuna (TPE)</code></td>
      <td style="padding:9px 14px">Bayesian, learns from past trials</td>
      <td style="padding:9px 14px">Most efficient for expensive models</td>
    </tr>
  </tbody>
</table>

### 6.1 Double-underscore parameter naming

Inside a Pipeline, hyperparameter names follow `step__param` syntax. For a nested Pipeline (like ColumnTransformer with sub-pipelines), add another `__`.

In [ ]:
# Examples of double-underscore naming in our pipe_full:
# pipe_full.set_params(model__C=0.1)            <- model's C parameter
# pipe_full.set_params(preprocessor__num__with_mean=False) <- scaler inside num sub-pipe

example_params: dict[str, Any] = pipe_full.get_params()
lr_keys: list[str] = [k for k in example_params if "model__" in k]
logger.info(f"Parameters accessible via double underscore: {lr_keys}")

### 6.2 Define the hyperparameter grid

We search over two parameters: the regularisation strength `C` and the solver `penalty` type.

In [ ]:
param_grid: dict[str, list[Any]] = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"],
}

### 6.3 Run GridSearchCV

`GridSearchCV` evaluates **every combination** in the grid -- 4 x 2 x 1 = 8 candidates here.

In [ ]:
grid_search: GridSearchCV = GridSearchCV(
    pipe_full,
    param_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring="f1",
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)
logger.info(f"Best grid params:  {grid_search.best_params_}")
logger.info(f"Best grid CV F1:   {grid_search.best_score_:.3f}")

### 6.4 Inspect all grid search results

In [ ]:
grid_df: pd.DataFrame = (
    pd.DataFrame(grid_search.cv_results_)[
        ["param_model__C", "param_model__penalty", "mean_test_score", "std_test_score", "rank_test_score"]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
    .rename(
        columns={
            "param_model__C": "C",
            "param_model__penalty": "penalty",
            "mean_test_score": "F1_mean",
            "std_test_score": "F1_std",
            "rank_test_score": "rank",
        }
    )
)
(
    themed_gt(GT(grid_df), n_rows=len(grid_df)).tab_header(
        title=gt_md("**GridSearchCV Results**"), subtitle="All candidate combinations"
    )
)

### 6.5 Define the distribution for RandomizedSearchCV

Instead of a fixed grid, we define probability distributions. `loguniform` samples log-uniformly over [1e-4, 1e2], which is appropriate for a regularisation parameter that spans orders of magnitude.

In [ ]:
param_dist: dict[str, Any] = {
    "model__C": loguniform(1e-4, 1e2),
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"],
}

### 6.6 Run RandomizedSearchCV with a fixed budget of 20 trials

In [ ]:
rand_search: RandomizedSearchCV = RandomizedSearchCV(
    pipe_full,
    param_dist,
    n_iter=20,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring="f1",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)
rand_search.fit(X_train, y_train)
logger.info(f"Best random params: {rand_search.best_params_}")
logger.info(f"Best random CV F1:  {rand_search.best_score_:.3f}")

### 6.7 Compare all search strategies

In [ ]:
search_compare: pd.DataFrame = pd.DataFrame(
    [
        {
            "Strategy": "Default",
            "Best_C": 1.0,
            "CV_F1": cv_scores.mean(),
        },
        {
            "Strategy": "GridSearchCV",
            "Best_C": grid_search.best_params_["model__C"],
            "CV_F1": grid_search.best_score_,
        },
        {
            "Strategy": "RandomizedSearchCV",
            "Best_C": rand_search.best_params_["model__C"],
            "CV_F1": rand_search.best_score_,
        },
    ]
)
metrics_report(
    search_compare,
    metrics=["Best_C", "CV_F1"],
    maximize_cols=["CV_F1"],
    title="Hyperparameter Search Comparison",
    subtitle="LogisticRegression on hotel booking cancellation",
)

::: {.callout-tip icon=false}
## <i class="bi bi-puzzle-fill" style="color:#059669"></i>&nbsp; Activity 3: search over RandomForest

Define a `param_dist` for `RandomForestClassifier` (`model__n_estimators`, `model__max_depth`, `model__min_samples_split`) and run `RandomizedSearchCV` with 10 trials.

```python
# Your code here
```
:::

## 7. Bayesian Optimisation with Optuna

**From Part 24 (ML Workflow, Section 7.3):** Optuna's TPE sampler builds a probabilistic model of which regions of the search space are promising and directs new trials toward them. It finds good hyperparameters with fewer evaluations than random search.

The three Optuna building blocks:
- **Trial:** one set of hyperparameters and the resulting score
- **Study:** a collection of trials with a direction (maximise or minimise)
- **Objective function:** takes a trial, builds a pipeline with those params, returns the score

### 7.1 Define the Optuna objective function

In [ ]:
def optuna_objective(trial: optuna.Trial) -> float:
    """Return 5-fold CV F1 for the hyperparameters suggested by trial."""
    C: float = trial.suggest_float("C", 1e-4, 1e2, log=True)
    penalty: str = trial.suggest_categorical("penalty", ["l1", "l2"])
    candidate: Pipeline = Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "model",
                LogisticRegression(
                    C=C,
                    penalty=penalty,
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    scores: np.ndarray = cross_val_score(
        candidate,
        X_train,
        y_train,
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        scoring="f1",
        n_jobs=-1,
    )
    return float(scores.mean())

### 7.2 Create the Optuna study

`direction="maximize"` tells Optuna to seek higher F1 scores.

In [ ]:
study: optuna.Study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

### 7.3 Run the optimisation

25 trials is enough to see convergence on this two-parameter problem. Silence Optuna's output; use `logger` after completion.

In [ ]:
study.optimize(optuna_objective, n_trials=25, show_progress_bar=False)
logger.success(
    f"Best Optuna trial: C={study.best_params['C']:.4f}  "
    f"penalty={study.best_params['penalty']}  "
    f"F1={study.best_value:.3f}"
)

### 7.4 Plot the optimisation history

Shows how the best F1 improved as Optuna explored the search space.

In [ ]:
trial_df: pd.DataFrame = pd.DataFrame(
    [
        {"trial": t.number, "f1": t.value, "best": max(t_.value for t_ in study.trials[: i + 1])}
        for i, t in enumerate(study.trials)
    ]
)
(
    ggplot(trial_df, aes(x="trial"))
    + geom_point(aes(y="f1"), color=INFO, alpha=0.5, size=2)
    + geom_line(aes(y="best"), color=SUCCESS, size=1.5)
    + labs(
        title="Optuna Optimisation History",
        subtitle="Blue dots = trial scores; green line = running best",
        x="Trial Number",
        y="CV F1 Score",
    )
    + modern_theme(grid=True)
)

### 7.5 Build and evaluate the best pipeline found by Optuna

In [ ]:
best_c: float = study.best_params["C"]
best_pen: str = study.best_params["penalty"]
pipe_optuna: Pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                C=best_c,
                penalty=best_pen,
                solver="liblinear",
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
pipe_optuna.fit(X_train, y_train)
logger.success("Best Optuna pipeline fitted on full training set")

### 7.6 Evaluate on the held-out test set

**Important:** the test set is evaluated only once -- after all hyperparameter search is complete.

In [ ]:
y_pred_optuna: np.ndarray = pipe_optuna.predict(X_test)
report_str: str = classification_report(y_test, y_pred_optuna, target_names=["Kept", "Cancelled"])
logger.info("\n" + report_str)

::: {.callout-tip icon=false}
## <i class="bi bi-puzzle-fill" style="color:#059669"></i>&nbsp; Activity 4: add max_iter to the Optuna search

Extend `optuna_objective` to also search over `max_iter` in `[200, 500, 1000, 2000]`.

```python
max_iter = trial.suggest_categorical('max_iter', [200, 500, 1000, 2000])
```

**Question:** does it change the optimal `C`?
:::

## 8. Feature Importance

**From Part 23 (What Is Machine Learning?, Section 5):** interpretability is part of the model contract. Coefficient magnitudes for a linear model reveal which features drive the prediction.

After `ColumnTransformer`, the raw feature indices no longer match our original column names -- we need `get_feature_names_out()` to reconstruct the mapping.

### 8.1 Define a helper to extract feature names from the Pipeline

In [ ]:
def get_feature_names(pipe: Pipeline) -> list[str]:
    """Return the full list of feature names after preprocessing."""
    ct: ColumnTransformer = pipe.named_steps["preprocessor"]
    return ct.get_feature_names_out().tolist()

### 8.2 Extract feature names for the Optuna pipeline

In [ ]:
fnames: list[str] = get_feature_names(pipe_optuna)
logger.info(f"Total features after preprocessing: {len(fnames)}")

### 8.3 Build a feature importance DataFrame

`coef_[0]` for a binary LogisticRegression has one value per feature. Positive = increases cancellation probability; negative = decreases.

In [ ]:
lr_model: LogisticRegression = pipe_optuna.named_steps["model"]
importance_df: pd.DataFrame = (
    pd.DataFrame(
        {
            "feature": fnames,
            "coefficient": lr_model.coef_[0],
        }
    )
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

### 8.4 Display the top features as a table

In [ ]:
(
    themed_gt(GT(importance_df[["feature", "coefficient", "abs_coef"]]), n_rows=len(importance_df)).tab_header(
        title=gt_md("**Top 20 Feature Importances (LogisticRegression)**"),
        subtitle="Coefficient magnitude = relative importance for cancellation prediction",
    )
)

### 8.5 Visualise the top 15 features

In [ ]:
plot_df: pd.DataFrame = importance_df.head(15).assign(
    color=lambda d: d["coefficient"].apply(lambda v: "positive" if v >= 0 else "negative")
)
(
    ggplot(
        plot_df,
        aes(
            x=as_discrete("feature", order_by="abs_coef", order=1),
            y="coefficient",
            fill="color",
        ),
    )
    + geom_bar(stat="identity", alpha=0.85)
    + scale_fill_manual(values={"positive": INFO, "negative": DANGER})
    + labs(
        title="Top 15 Features -- LogisticRegression Coefficients",
        subtitle="Positive = increases cancellation probability",
        x="",
        y="Coefficient",
    )
    + modern_theme(x_axis_angle=40, legend_pos="none")
)

::: {.callout-tip icon=false}
## <i class="bi bi-lightbulb-fill" style="color:#8B5CF6"></i>&nbsp; Pro Tip: feature importance sanity check

Cross-check the top features against domain knowledge. `deposit_type=Non Refund` having a large positive coefficient makes intuitive sense: non-refundable deposits are associated with bookings that are not cancelled. If a feature ranks highly but has no obvious interpretation, investigate for data leakage.
:::

## 9. Final Evaluation and Model Card

**From Part 13 (Dev Tools, Section 5 -- Reproducibility):** a model card documents who trained the model, on what data, with what performance, so the model artefact is not a black box.

### 9.1 Compute final CV metrics on the best pipeline

In [ ]:
final_cv: np.ndarray = cross_val_score(
    pipe_optuna,
    X_train,
    y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring="f1",
    n_jobs=-1,
)
test_f1: float = f1_score(y_test, y_pred_optuna)
logger.info(f"Final CV F1 (5-fold): {final_cv.mean():.3f} +/- {final_cv.std():.3f}")
logger.info(f"Test F1 (held-out):   {test_f1:.3f}")

### 9.2 Build the model card dictionary

In [ ]:
model_card: dict[str, Any] = {
    "model_class": "LogisticRegression",
    "target": TARGET_CLF,
    "features": ALL_FEATURES,
    "n_train": len(X_train),
    "n_test": len(X_test),
    "search_strategy": "Optuna TPE (25 trials)",
    "best_c": best_c,
    "best_penalty": best_pen,
    "cv_f1_mean": float(final_cv.mean()),
    "cv_f1_std": float(final_cv.std()),
    "test_f1": test_f1,
    "dataset": "Hotel Booking Demand (Antonio et al., 2019)",
}
for k, v in model_card.items():
    logger.info(f"  {k:<22} {v}")

### 9.3 Save the full pipeline bundle

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
bundle_path: Path = MODEL_DIR / "cancellation_pipeline.joblib"
joblib.dump(
    {
        "pipeline": pipe_optuna,
        "features": ALL_FEATURES,
        "model_card": model_card,
    },
    bundle_path,
)
logger.success(f"Bundle saved to {bundle_path}")

### 9.4 Reload and verify on one booking

In [ ]:
loaded: dict[str, Any] = joblib.load(bundle_path)
sample_row: pd.DataFrame = X_test.iloc[:1]
prediction: int = int(loaded["pipeline"].predict(sample_row)[0])
actual: int = int(y_test[0])
logger.info(f"Predicted: {prediction}  Actual: {actual}  ({'match' if prediction == actual else 'mismatch'})")

## Capstone: Full Classification Pipeline Function

Encapsulate the complete workflow -- load, split, build pipeline, search with Optuna, evaluate, save -- in a single typed function.

### Define the capstone function

In [ ]:
def build_classification_pipeline(
    df: pd.DataFrame,
    numerical_features: list[str],
    categorical_features: list[str],
    target: str,
    n_trials: int = 25,
    out_dir: Path = Path("models"),
) -> dict[str, Any]:
    """Build, tune, and save a classification pipeline. Returns metrics and artefact paths."""
    all_feats = numerical_features + categorical_features
    X_all: pd.DataFrame = df[all_feats].copy()
    y_all: np.ndarray = df[target].to_numpy(dtype=int)

    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all)

    num_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    cat_pipe = make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    )
    prep = ColumnTransformer(
        [
            ("num", num_pipe, numerical_features),
            ("cat", cat_pipe, categorical_features),
        ]
    )

    def _objective(trial: optuna.Trial) -> float:
        C = trial.suggest_float("C", 1e-4, 1e2, log=True)
        pen = trial.suggest_categorical("penalty", ["l1", "l2"])
        pipe = Pipeline(
            [
                ("preprocessor", prep),
                (
                    "model",
                    LogisticRegression(
                        C=C,
                        penalty=pen,
                        solver="liblinear",
                        class_weight="balanced",
                        max_iter=1000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )
        scores = cross_val_score(
            pipe,
            Xtr,
            ytr,
            cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
            scoring="f1",
            n_jobs=-1,
        )
        return float(scores.mean())

    study_inner = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study_inner.optimize(_objective, n_trials=n_trials, show_progress_bar=False)

    best = study_inner.best_params
    final_pipe = Pipeline(
        [
            ("preprocessor", prep),
            (
                "model",
                LogisticRegression(
                    C=best["C"],
                    penalty=best["penalty"],
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    final_pipe.fit(Xtr, ytr)

    test_f1: float = f1_score(yte, final_pipe.predict(Xte))
    out_dir.mkdir(exist_ok=True)
    path: Path = out_dir / f"{target}_pipeline.joblib"
    joblib.dump({"pipeline": final_pipe, "features": all_feats}, path)
    logger.success(f"Saved pipeline bundle to {path}")
    logger.info(f"Test F1: {test_f1:.3f}  Best params: {best}")
    return {"pipeline": final_pipe, "test_f1": test_f1, "best_params": best, "path": path}

### Run the capstone

In [ ]:
# Clean up any previous model artefacts before re-running
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)

result: dict[str, Any] = build_classification_pipeline(
    df,
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_CLF,
    n_trials=25,
    out_dir=MODEL_DIR,
)

## Further Reading

- [Scikit-learn Pipeline docs](https://scikit-learn.org/stable/modules/compose.html) -- ColumnTransformer + Pipeline reference
- [Optuna documentation](https://optuna.readthedocs.io/) -- TPE sampler, pruning, visualisation
- [Optuna Paper](https://arxiv.org/abs/1907.10902) -- Akiba et al. (2019), framework design and benchmarks
- Geron (2022), *Hands-On Machine Learning*, Ch 2 -- pipeline and hyperparameter search walkthrough
- Antonio et al. (2019), *Hotel booking demand datasets*, Data in Brief -- dataset reference

**Next:** Part 5 (Deep Learning) will apply the same Pipeline concept to neural networks with scikit-learn-compatible wrappers.